# 3-D Benchmark — Prompt U-Net vs nnInteractive

**Combined runner + results notebook.**

1. Configure paths and run parameters in **Section 1**.
2. Run the benchmark in **Section 2** (or skip to Section 3 to load existing results).
3. Analyse and visualise results in **Section 3**.

---
## Section 1 — Configuration

Edit the variables below before running anything else.

In [ ]:
import sys, os

# ── Project root (adjust if notebook is moved) ──────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Model paths ─────────────────────────────────────────────────────────────
P_UNET_MODELS = [
    os.path.join(PROJECT_ROOT, 'training', 'p_unet_313.keras'),
    # add more paths here to evaluate multiple versions ...
]
NN_MODEL_DIR   = None    # None = auto-download from HuggingFace Hub

# ── Dataset .npz paths (ds_handler format) ──────────────────────────────────
NPZ_PATHS = [
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_ct.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_mri.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'SegRap2023.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'HCCTase_ceCT.npz'),
]

# ── Run parameters ──────────────────────────────────────────────────────────
RUNS_PER_VOL       = 5         # random prompts evaluated per volume
# MODES supported: 'ifl_ssf' (GT feedback + SSF), 'ifl' (GT only), 'ssf' (SSF only), 'none' (neither)
MODES              = ['ifl', 'ifl_ssf']   
MODALITY           = None      # important to use, when modality is not saved in the test .npz 
OUTPUT_THRESHOLD   = 0.5       # sigmoid threshold for P-UNet binary mask
SIM_DIFF_THRESHOLD = 0.6       # SSIM drop that triggers SSF refresh
GT_DICE_THRESHOLD  = 0.65      # IFL Dice threshold for GT correction
BATCH_SIZE         = 3         # slices per GPU forward pass 
WINDOW             = 10        # half-width for windowed Dice evaluation
MIN_PROMPT_PIXELS  = 50        # minimum foreground pixels for a prompt to be eligible
NN_DEVICE          = None      # None = auto-detect (CUDA if available, else CPU)

# ── Volumes to evaluate ─────────────────────────────────────────────────────
MAX_VOLUMES        = None      # int (e.g. 3) to test subset, or None to evaluate all volumes

# ── Output directory (pkl + json are saved here) ────────────────────────────
OUTPUT_DIR = os.path.join(os.getcwd())

print('Configuration loaded.')
print(f'  Models: {[os.path.basename(m) for m in P_UNET_MODELS]}')
print(f'  Modes : {MODES}')
print(f'  npz   : {[os.path.basename(p) for p in NPZ_PATHS]}')
print(f'  Max Volumes: {MAX_VOLUMES if MAX_VOLUMES else "All"}')
print(f'  Output: {OUTPUT_DIR}')


Configuration loaded.
  Model : c:\Users\Paul\Desktop\Studium\prompt-unet\training\p_unet_313.keras
  Mode  : ifl
  npz   : ['c:\\Users\\Paul\\Desktop\\Studium\\prompt-unet\\data\\test_data\\Mouse_no_trachea.npz']
  Output: c:\Users\Paul\Desktop\Studium\prompt-unet\evaluation\benchmark_nninteractive


---
## Section 2 — Run Benchmark

Runs Prompt-UNet and nnInteractive on all configured datasets.  
Results are saved automatically to `OUTPUT_DIR` as `results_<timestamp>.pkl` and `results_<timestamp>_summary.json`.  

> ⚠️ **Skip this section** if you already have a `.pkl` file and only want to analyse results.

In [ ]:
from evaluation.benchmark_nninteractive.benchmark_3d import run_benchmark
import os

all_records = []
for p_model in P_UNET_MODELS:
    for mode in MODES:
        print(f"\n--- Running: {os.path.basename(p_model)} | mode: {mode} ---")
        records = run_benchmark(
            npz_paths           = NPZ_PATHS,
            p_unet_model        = p_model,
            nn_model_dir        = NN_MODEL_DIR,
            runs_per_vol        = RUNS_PER_VOL,
            mode                = mode,
            modality            = MODALITY,
            output_threshold    = OUTPUT_THRESHOLD,
            sim_diff_threshold  = SIM_DIFF_THRESHOLD,
            gt_dice_threshold   = GT_DICE_THRESHOLD,
            window              = WINDOW,
            min_prompt_pixels   = MIN_PROMPT_PIXELS,
            max_volumes         = MAX_VOLUMES,
            output_dir          = OUTPUT_DIR,
            nn_device           = NN_DEVICE,
            verbose             = True,
            return_predictions  = False, # Kept false to avoid memory bloat
        )
        all_records.extend(records)

print(f"\nAll benchmark loops complete — {len(all_records)} total runs collected.")


---
## Section 3 — Load & Analyse Results

Either `records` from Section 2 is already in memory, or load an existing `.pkl` file below.

In [ ]:
import pickle, glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Option A: use results already in memory from Section 2 ──────────────────
# records = records   # already set above — nothing to do

# ── Option B: load the most-recent pkl from OUTPUT_DIR ──────────────────────
pkl_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'results_*.pkl')))
if not pkl_files:
    raise FileNotFoundError(f'No results .pkl found in {OUTPUT_DIR}')
LOAD_PATH = pkl_files[-1]   # ← change to a specific filename if needed
print(f'Loading: {LOAD_PATH}')
with open(LOAD_PATH, 'rb') as f:
    records = pickle.load(f)

# Build a DataFrame for easy analysis
df = pd.DataFrame(records)
print(f'Loaded {len(df)} runs.')
df.head()

### 3.1 — Overall summary statistics

In [ ]:
def _fmt(s):
    return f'{s.mean():.3f} ± {s.std():.3f}  [median {s.median():.3f},  min {s.min():.3f},  max {s.max():.3f}]'

mode = df['mode'].iloc[0] if len(df) else '—'
n    = len(df)

print(f'{'='*68}')
print(f'  RESULTS SUMMARY   mode={mode}   n={n}')
print(f'{'='*68}')
print(f'  Prompt-UNet')
print(f'    Vol Dice    : {_fmt(df["p_unet_vol_dice"])}')
print(f'    Window Dice : {_fmt(df["p_unet_window_dice"])}')

if df['num_user_interacts'].notna().any():
    coli = df['num_user_interacts'].dropna()
    print(f'    User interactions (incl. initial): {_fmt(coli)}')

print(f'  nnInteractive')
print(f'    Vol Dice    : {_fmt(df["nn_vol_dice"])}')
print(f'    Window Dice : {_fmt(df["nn_window_dice"])}')
print(f'{'='*68}')

### 3.2 — Per-volume breakdown

In [ ]:
agg = df.groupby('volume_id').agg(
    n_runs              = ('p_unet_vol_dice', 'count'),
    p_unet_vol_mean     = ('p_unet_vol_dice',    'mean'),
    p_unet_vol_std      = ('p_unet_vol_dice',    'std'),
    p_unet_win_mean     = ('p_unet_window_dice', 'mean'),
    nn_vol_mean         = ('nn_vol_dice',        'mean'),
    nn_vol_std          = ('nn_vol_dice',        'std'),
    nn_win_mean         = ('nn_window_dice',     'mean'),
).round(3)

agg['p_unet_vol']  = agg.apply(lambda r: f"{r['p_unet_vol_mean']:.3f} ±{r['p_unet_vol_std']:.3f}", axis=1)
agg['nn_vol']      = agg.apply(lambda r: f"{r['nn_vol_mean']:.3f} ±{r['nn_vol_std']:.3f}",     axis=1)

display_cols = ['n_runs', 'p_unet_vol', 'p_unet_win_mean', 'nn_vol', 'nn_win_mean']
agg[display_cols].rename(columns={
    'p_unet_win_mean': 'p_unet_win',
    'nn_win_mean':     'nn_win',
})

### 3.3 — Dice distribution boxplots

In [ ]:
def _boxplot(ax, data_list, labels, title, colors):
    bp = ax.boxplot(
        data_list,
        tick_labels   = labels,
        widths        = 0.5,
        patch_artist  = True,
        boxprops      = dict(linewidth=1.2),
        medianprops   = dict(color='white', linewidth=2),
        whiskerprops  = dict(linewidth=1),
        capprops      = dict(linewidth=1),
        flierprops    = dict(marker='o', markersize=4, alpha=0.5),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Dice')
    ax.grid(axis='y', alpha=0.3)

COLORS = ['#4A90D9', '#E87040']   # blue = P-UNet, orange = nnInteractive
LABELS = ['Prompt-UNet', 'nnInteractive']

fig, axes = plt.subplots(1, 2, figsize=(8, 4.5), sharey=True)

_boxplot(
    axes[0],
    [df['p_unet_vol_dice'].tolist(), df['nn_vol_dice'].tolist()],
    LABELS,
    'Volumetric Dice',
    COLORS,
)
_boxplot(
    axes[1],
    [df['p_unet_window_dice'].tolist(), df['nn_window_dice'].tolist()],
    LABELS,
    'Window Dice (±10 slices)',
    COLORS,
)

fig.suptitle(f'Prompt-UNet vs nnInteractive  –  {mode} mode  (n={n})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_boxplots.pdf'), bbox_inches='tight')
plt.show()
print('Figure saved to results/dice_boxplots.pdf')

### 3.4 — Scatter: P-UNet vs nnInteractive Dice (per run)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(
    df['p_unet_vol_dice'],
    df['nn_vol_dice'],
    alpha=0.6, s=30, color='#4A90D9', edgecolors='none',
)
# Diagonal reference line
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Equal performance')

ax.set_xlabel('Prompt-UNet  volumetric Dice', fontsize=11)
ax.set_ylabel('nnInteractive  volumetric Dice', fontsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
ax.set_title(f'Per-run scatter  (n={n})', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_scatter.pdf'), bbox_inches='tight')
plt.show()

### 3.5 — IFL interaction distribution  *(IFL mode only)*

In [ ]:
ifl_df = df[df['num_user_interacts'].notna()]

if ifl_df.empty:
    print('No IFL data found (run in SSF mode or interactions not recorded).')
else:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    counts = ifl_df['num_user_interacts'].astype(int)
    bins   = range(counts.min(), counts.max() + 2)
    ax.hist(counts, bins=bins, align='left', rwidth=0.7, color='#4A90D9', edgecolor='white')
    ax.set_xlabel('User interactions (incl. initial prompt)', fontsize=11)
    ax.set_ylabel('Number of runs', fontsize=11)
    ax.set_title(f'IFL interaction count distribution  (mean={counts.mean():.1f})', fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ifl_interactions.pdf'), bbox_inches='tight')
    plt.show()

### 3.6 — Dice by prompt axis

In [ ]:
axis_labels = {0: 'Axial (0)', 1: 'Coronal (1)', 2: 'Sagittal (2)'}
present_axes = sorted(df['prompt_axis'].unique())

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)

for metric_col, ax, title in [
    ('p_unet_vol_dice', axes[0], 'Prompt-UNet  Vol Dice'),
    ('nn_vol_dice',     axes[1], 'nnInteractive  Vol Dice'),
]:
    grouped = [df[df['prompt_axis'] == a][metric_col].tolist() for a in present_axes]
    ax.boxplot(
        grouped,
        tick_labels   = [axis_labels.get(a, str(a)) for a in present_axes],
        widths        = 0.5,
        patch_artist  = True,
        boxprops      = dict(facecolor='#cce0f5', linewidth=1.2),
        medianprops   = dict(color='#1a5f9e', linewidth=2),
        whiskerprops  = dict(linewidth=1),
        capprops      = dict(linewidth=1),
    )
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Dice')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Dice by prompt axis', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_by_axis.pdf'), bbox_inches='tight')
plt.show()

### 3.7 — Dice by ROI label

In [ ]:
roi_groups = df.groupby('selected_roi')[['p_unet_vol_dice', 'nn_vol_dice']].mean().round(3)
roi_groups.columns = ['P-UNet  vol', 'nnInteractive  vol']
roi_groups.index.name = 'ROI label'
roi_groups

---
## Section 4 — Export DataFrame

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, 'results_flat.csv')
df.drop(columns=['user_interacts_idx'], errors='ignore').to_csv(csv_path, index=False)
print(f'DataFrame exported to {csv_path}')